In [1]:
# ════════════════════════════════════════════
# CELL 1 — Setup
# ════════════════════════════════════════════
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

from src.phase4_nerf.model   import NeRFModel, NeRFSmall, PositionalEncoding
from src.phase4_nerf.render  import render_rays, sample_along_rays, volume_render
from src.phase4_nerf.evaluate import compute_psnr, compute_ssim, render_full_image
from src.phase4_nerf.utils   import (
    generate_360_path, normalize_scene,
    visualize_rays, visualize_depth_map,
    CheckpointManager
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("../outputs/nerf_checkpoints", exist_ok=True)
os.makedirs("../outputs/renders", exist_ok=True)

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Device: cuda
PyTorch: 2.5.1+cu121
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM: 4.0 GB


In [2]:
# ════════════════════════════════════════════
# CELL 2 — Dataset Loading
# ════════════════════════════════════════════
SCENE_DIR = "../data/nerf_synthetic/lego"

try:
    from src.phase4_nerf.dataset import NeRFSyntheticDataset
    train_ds = NeRFSyntheticDataset(SCENE_DIR, split="train", img_scale=0.25)
    val_ds   = NeRFSyntheticDataset(SCENE_DIR, split="val",   img_scale=0.25)
    DATASET_LOADED = True
    print(f"Dataset loaded: {len(train_ds.images)} train, {len(val_ds.images)} val images")
    print(f"Image size: {train_ds.W}x{train_ds.H}, focal={train_ds.focal:.1f}")
    near, far = 2.0, 6.0
except Exception as e:
    print(f"Dataset not found: {e}")
    print("Generating synthetic dataset for demo...")
    DATASET_LOADED = False

    # ── Synthetic "scene": colored sphere ──────────────────
    class SyntheticSphereDset:
        def __init__(self, n_views=50, H=64, W=64):
            self.H, self.W = H, W
            self.focal = 50.0
            self.K = np.array([[self.focal, 0, W/2],
                               [0, self.focal, H/2],
                               [0, 0, 1]], dtype=np.float64)
            self._build(n_views)

        def _build(self, n_views):
            from src.phase4_nerf.dataset import get_rays
            from src.utils.transforms import rotation_matrix_y

            all_rays_o, all_rays_d, all_rgb = [], [], []
            self.images = []
            self.poses  = []

            for i in range(n_views):
                angle = i * (360.0 / n_views)
                R = rotation_matrix_y(angle)
                t = np.array([0, 0, 3.0])
                c2w = np.eye(4)
                c2w[:3, :3] = R.T
                c2w[:3, 3]  = R.T @ (-t)
                self.poses.append(c2w)

                # Render a synthetic colored sphere
                img = self._render_sphere(c2w)
                self.images.append(img)

                ro, rd = get_rays(self.H, self.W, self.K, c2w)
                all_rays_o.append(ro.reshape(-1, 3))
                all_rays_d.append(rd.reshape(-1, 3))
                all_rgb.append(img.reshape(-1, 3))

            self.all_rays_o = torch.FloatTensor(np.concatenate(all_rays_o))
            self.all_rays_d = torch.FloatTensor(np.concatenate(all_rays_d))
            self.all_rgb    = torch.FloatTensor(np.concatenate(all_rgb))
            self.images     = np.stack(self.images)
            self.poses      = np.stack(self.poses)

        def _render_sphere(self, c2w):
            from src.phase4_nerf.dataset import get_rays
            img = np.ones((self.H, self.W, 3), dtype=np.float32)
            cx, cy = self.W / 2, self.H / 2
            r = min(self.W, self.H) * 0.3
            for y in range(self.H):
                for x in range(self.W):
                    if (x - cx)**2 + (y - cy)**2 < r**2:
                        nx = (x - cx) / r
                        ny = (y - cy) / r
                        nz = np.sqrt(max(0, 1 - nx**2 - ny**2))
                        img[y, x] = np.clip(
                            [(nx + 1)/2, (ny + 1)/2, (nz + 1)/2], 0, 1
                        )
            return img

        def __len__(self):
            return len(self.all_rays_o)

        def __getitem__(self, idx):
            return {
                "rays_o": self.all_rays_o[idx],
                "rays_d": self.all_rays_d[idx],
                "rgb":    self.all_rgb[idx],
            }

        def get_image_rays(self, img_idx):
            from src.phase4_nerf.dataset import get_rays
            ro, rd = get_rays(self.H, self.W, self.K, self.poses[img_idx])
            return {
                "rays_o": torch.FloatTensor(ro),
                "rays_d": torch.FloatTensor(rd),
                "rgb":    torch.FloatTensor(self.images[img_idx]),
            }

    train_ds = SyntheticSphereDset(n_views=50)
    val_ds   = SyntheticSphereDset(n_views=10)
    near, far = 1.0, 5.0
    print(f"Synthetic dataset: {len(train_ds.images)} views, {train_ds.H}x{train_ds.W}")

Loaded 100 train images (200x200), focal=277.8
Loaded 100 val images (200x200), focal=277.8
Dataset loaded: 100 train, 100 val images
Image size: 200x200, focal=277.8


In [3]:


# ════════════════════════════════════════════
# CELL 3 — Visualize Training Data
# ════════════════════════════════════════════
n_show = min(8, len(train_ds.images))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < n_show:
        img = np.clip(train_ds.images[i], 0, 1)
        ax.imshow(img)
        ax.set_title(f"View {i}", fontsize=9)
    ax.axis("off")
plt.suptitle(f"Training Views ({len(train_ds.images)} total)", fontsize=13)
plt.tight_layout()
plt.savefig("../outputs/nerf_checkpoints/training_views.png", dpi=120)
plt.show()


C:\Users\Harsh\AppData\Local\Temp\ipykernel_3316\594574969.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:

# ════════════════════════════════════════════
# CELL 4 — Ray Visualization
# ════════════════════════════════════════════
sample_data = train_ds.get_image_rays(0)
rays_o_np   = sample_data["rays_o"].numpy()
rays_d_np   = sample_data["rays_d"].numpy()

if len(rays_o_np.shape) == 3:
    rays_o_np = rays_o_np.reshape(-1, 3)
    rays_d_np = rays_d_np.reshape(-1, 3)

fig = visualize_rays(
    rays_o_np, rays_d_np,
    n_samples=32,
    near=near, far=far,
    max_rays=15,
    output_path="../outputs/nerf_checkpoints/ray_visualization.png"
)
plt.show()
plt.close()


Saved ray visualization: ../outputs/nerf_checkpoints/ray_visualization.png


C:\Users\Harsh\AppData\Local\Temp\ipykernel_3316\3377196720.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:

# ════════════════════════════════════════════
# CELL 5 — Model Architecture
# ════════════════════════════════════════════
from src.phase4_nerf.utils import print_model_summary

# Quick model for notebook demo
model_coarse = NeRFSmall(pos_enc_dims=6, dir_enc_dims=2, hidden_dim=64).to(device)
model_fine   = NeRFSmall(pos_enc_dims=6, dir_enc_dims=2, hidden_dim=64).to(device)

print_model_summary(model_coarse, [(4096, 3), (4096, 3)])

total_params = sum(p.numel() for p in model_coarse.parameters()) * 2
print(f"\nTotal (coarse + fine): {total_params:,} parameters")
print(f"Model size: ~{total_params * 4 / 1024 / 1024:.1f} MB (float32)")


Model: NeRFSmall
Layer                          Output Shape             Params
--------------------------------------------------------------
pos_encoding                   PositionalEncoding            0
dir_encoding                   PositionalEncoding            0
net.0                          Linear                    2,560
net.1                          ReLU                          0
net.2                          Linear                    4,160
net.3                          ReLU                          0
net.4                          Linear                    4,160
net.5                          ReLU                          0
net.6                          Linear                    4,160
net.7                          ReLU                          0
density_out                    Linear                       65
feature_out                    Linear                    4,160
color_net.0                    Linear                    2,560
color_net.1                    ReLU  

In [6]:


# ════════════════════════════════════════════
# CELL 6 — Training Loop (Quick Demo)
# ════════════════════════════════════════════
optimizer  = torch.optim.Adam(
    list(model_coarse.parameters()) + list(model_fine.parameters()),
    lr=5e-4
)
scheduler  = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2000, gamma=0.5)
BATCH_SIZE = 512
N_ITERS    = 500   # Quick demo; use 50000+ for real quality

train_losses = []
val_psnrs    = []
n_rays       = len(train_ds)

print(f"Quick training demo: {N_ITERS} iterations, batch={BATCH_SIZE}")
print(f"(For real quality: 50,000-200,000 iterations — use scripts/train_nerf.py)")

for iteration in range(1, N_ITERS + 1):
    model_coarse.train(); model_fine.train()

    indices = torch.randint(0, n_rays, (BATCH_SIZE,))
    rays_o  = train_ds.all_rays_o[indices].to(device)
    rays_d  = train_ds.all_rays_d[indices].to(device)
    rgb_gt  = train_ds.all_rgb[indices].to(device)

    result = render_rays(
        model=model_coarse,
        rays_o=rays_o, rays_d=rays_d,
        near=near, far=far,
        num_coarse=32, num_fine=32,
        model_fine=model_fine,
        perturb=True, white_background=True
    )

    loss = torch.nn.functional.mse_loss(result["rgb_coarse"], rgb_gt)
    if "rgb_fine" in result:
        loss += torch.nn.functional.mse_loss(result["rgb_fine"], rgb_gt)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    train_losses.append(loss.item())

    if iteration % 100 == 0:
        psnr = -10 * np.log10(loss.item())
        lr   = optimizer.param_groups[0]["lr"]
        print(f"  Iter {iteration:4d} | Loss: {loss.item():.6f} | PSNR: {psnr:.2f}dB | LR: {lr:.2e}")

print("\nQuick demo training complete!")

Quick training demo: 500 iterations, batch=512
(For real quality: 50,000-200,000 iterations — use scripts/train_nerf.py)
  Iter  100 | Loss: 0.172349 | PSNR: 7.64dB | LR: 5.00e-04
  Iter  200 | Loss: 0.167412 | PSNR: 7.76dB | LR: 5.00e-04
  Iter  300 | Loss: 0.170751 | PSNR: 7.68dB | LR: 5.00e-04
  Iter  400 | Loss: 0.178689 | PSNR: 7.48dB | LR: 5.00e-04
  Iter  500 | Loss: 0.156598 | PSNR: 8.05dB | LR: 5.00e-04

Quick demo training complete!


In [7]:


# ════════════════════════════════════════════
# CELL 7 — Loss Curve
# ════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(train_losses, color="steelblue", linewidth=0.8, alpha=0.7)

# Smoothed curve
if len(train_losses) > 20:
    w = 20
    smooth = np.convolve(train_losses, np.ones(w)/w, mode="valid")
    ax.semilogy(
        range(w//2, w//2 + len(smooth)), smooth,
        color="red", linewidth=2.0, label=f"Smoothed (w={w})"
    )
    ax.legend()

ax.set_xlabel("Iteration"); ax.set_ylabel("MSE Loss (log)"); ax.set_title("Training Loss Curve")
ax.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.savefig("../outputs/nerf_checkpoints/loss_curve.png", dpi=120)
plt.show()

C:\Users\Harsh\AppData\Local\Temp\ipykernel_3316\626708234.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:


# ════════════════════════════════════════════
# CELL 8 — Render Validation Image
# ════════════════════════════════════════════
model_coarse.eval(); model_fine.eval()

val_data = val_ds.get_image_rays(0)
rays_o_v = val_data["rays_o"].to(device)
rays_d_v = val_data["rays_d"].to(device)

if len(rays_o_v.shape) == 3:
    H_v, W_v = rays_o_v.shape[:2]
else:
    H_v = W_v = int(np.sqrt(len(rays_o_v)))

rgb_pred, depth_pred = render_full_image(
    model_coarse, model_fine,
    rays_o_v, rays_d_v,
    H_v, W_v,
    near=near, far=far,
    num_coarse=32, num_fine=32,
    chunk_size=4096
)

rgb_gt = val_data["rgb"].numpy().reshape(H_v, W_v, 3)
psnr   = compute_psnr(rgb_pred, rgb_gt)
ssim   = compute_ssim(rgb_pred, rgb_gt)

print(f"Validation metrics (after {N_ITERS} quick-demo iters):")
print(f"  PSNR : {psnr:.2f} dB  (target: 28+ dB with full training)")
print(f"  SSIM : {ssim:.4f}     (target: 0.90+ with full training)")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(np.clip(rgb_gt,   0, 1)); axes[0].set_title("Ground Truth");      axes[0].axis("off")
axes[1].imshow(np.clip(rgb_pred, 0, 1)); axes[1].set_title(f"NeRF Prediction\nPSNR={psnr:.2f}dB, SSIM={ssim:.3f}"); axes[1].axis("off")
err = np.abs(rgb_gt - rgb_pred).mean(axis=-1)
im  = axes[2].imshow(err, cmap="hot", vmin=0)
axes[2].set_title("L1 Error Map"); axes[2].axis("off")
plt.colorbar(im, ax=axes[2], fraction=0.046)
plt.suptitle(f"NeRF Rendering Quality (demo — {N_ITERS} iters)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/renders/nb04_validation.png", dpi=150)
plt.show()



Validation metrics (after 500 quick-demo iters):
  PSNR : 10.78 dB  (target: 28+ dB with full training)
  SSIM : 0.5897     (target: 0.90+ with full training)


C:\Users\Harsh\AppData\Local\Temp\ipykernel_3316\3426840183.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# ════════════════════════════════════════════
# CELL 9 — Novel View Synthesis (360 path)
# ════════════════════════════════════════════
from src.phase4_nerf.dataset import get_rays as get_rays_fn

n_frames = 24
path_poses = generate_360_path(radius=3.0, height=0.3, n_frames=n_frames)

K_render = np.array([
    [train_ds.focal, 0, train_ds.W/2],
    [0, train_ds.focal, train_ds.H/2],
    [0, 0, 1]
], dtype=np.float64) if hasattr(train_ds, "focal") else np.array(
    [[50, 0, 32], [0, 50, 32], [0, 0, 1]], dtype=np.float64
)

H_r = getattr(train_ds, "H", 64)
W_r = getattr(train_ds, "W", 64)

frames    = []
frame_dir = "../outputs/renders/frames"
os.makedirs(frame_dir, exist_ok=True)

print(f"Rendering {n_frames} novel views...")
for i, c2w in enumerate(path_poses):
    ro, rd = get_rays_fn(H_r, W_r, K_render, c2w)
    ro_t   = torch.FloatTensor(ro).to(device)
    rd_t   = torch.FloatTensor(rd).to(device)

    with torch.no_grad():
        rgb_f, _ = render_full_image(
            model_coarse, model_fine,
            ro_t, rd_t, H_r, W_r,
            near=near, far=far,
            num_coarse=32, num_fine=32,
            chunk_size=4096
        )

    frame = (np.clip(rgb_f, 0, 1) * 255).astype(np.uint8)
    frames.append(frame)
    Image.fromarray(frame).save(os.path.join(frame_dir, f"frame_{i:04d}.png"))

    if (i + 1) % 6 == 0:
        print(f"  Rendered {i+1}/{n_frames} frames")

# Display frame grid
fig, axes = plt.subplots(3, 8, figsize=(20, 8))
for i, ax in enumerate(axes.flat):
    if i < len(frames):
        ax.imshow(frames[i])
        ax.set_title(f"f{i}", fontsize=7)
    ax.axis("off")
plt.suptitle(f"Novel View Synthesis — 360° Orbit ({n_frames} frames)", fontsize=13)
plt.tight_layout()
plt.savefig("../outputs/renders/novel_views_grid.png", dpi=150)
plt.show()

print(f"\nSaved {n_frames} frames to {frame_dir}/")
print("To make a video: python -c \"from src.utils.io_utils import images_to_video; "
      "import glob; images_to_video(sorted(glob.glob('../outputs/renders/frames/*.png')), "
      "'../outputs/renders/orbit.mp4', fps=12)\"")



Rendering 24 novel views...
  Rendered 6/24 frames
  Rendered 12/24 frames
  Rendered 18/24 frames
  Rendered 24/24 frames

Saved 24 frames to ../outputs/renders/frames/
To make a video: python -c "from src.utils.io_utils import images_to_video; import glob; images_to_video(sorted(glob.glob('../outputs/renders/frames/*.png')), '../outputs/renders/orbit.mp4', fps=12)"


C:\Users\Harsh\AppData\Local\Temp\ipykernel_3316\789006113.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ════════════════════════════════════════════
# CELL 10 — Save Checkpoint
# ════════════════════════════════════════════
ckpt_mgr = CheckpointManager("../outputs/nerf_checkpoints", keep_last=3)
ckpt_path = ckpt_mgr.save(
    iteration=N_ITERS,
    model_coarse=model_coarse,
    model_fine=model_fine,
    optimizer=optimizer,
    scheduler=scheduler,
    metrics={"psnr": psnr, "ssim": ssim},
    config={
        "near": near, "far": far,
        "num_coarse": 32, "num_fine": 32,
        "batch_size": BATCH_SIZE,
    }
)

print(f"\nCheckpoint saved: {ckpt_path}")
print("\n── To continue training with full quality ──────────────────")
print("python scripts/train_nerf.py data/nerf_synthetic/lego")
print("\n── To render with trained model ────────────────────────────")
print("python scripts/render_views.py outputs/nerf_checkpoints/lego/ckpt_0050000.pt data/nerf_synthetic/lego")

  Saved checkpoint: ../outputs/nerf_checkpoints\ckpt_0000500.pt

Checkpoint saved: ../outputs/nerf_checkpoints\ckpt_0000500.pt

── To continue training with full quality ──────────────────
python scripts/train_nerf.py data/nerf_synthetic/lego

── To render with trained model ────────────────────────────
python scripts/render_views.py outputs/nerf_checkpoints/lego/ckpt_0050000.pt data/nerf_synthetic/lego
